# Phylogenetic Placement of *Lactobacillus* Species Using nf-core/phyloplace

## Table of Contents
- [Introduction](#introduction)
- [Research Question & Hypothesis](#research-question--hypothesis)
- [Analysis and Methods](#analysis-and-methods)
- [Results and Visualizations](#results-and-visualizations)
- [Discussion](#discussion)
- [Future Plan](#future-plan)
- [Limitations & Caveats](#limitations--caveats)
- [References](#references)

## Introduction

This notebook guides you through reproducing the phylogenetic placement of *Lactobacillus* query sequences onto a known reference tree using the **nf-core/phyloplace** workflow.

*Lactobacilli* are bacteria that colonize human and animal body sites. They are common probiotics found in fermented foods.

The **nf-core/phyloplace** pipeline performs phylogenetic placement using EPA-NG and containers (Docker/Singularity) to ensure reproducibility. We analyze three *Lactobacillus* genomes using this pipeline.

## Requirements

- **Nextflow installed**:
```bash
wget -qO nextflow https://github.com/nextflow-io/nextflow/releases/download/v22.10.0/nextflow
chmod +x nextflow
./nextflow -version
```
- **Docker** for container execution
- **Prepared input files**:
  - Reference MSA
  - Reference tree
  - Query FASTA sequences

## Analysis and Methods

### Step 1: Genomic Data Search
```bash
wget ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR485/ERR485020/ERR485020.fastq.gz
```

### Step 2: Genome Annotation Using Prokka
```bash
prokka --outdir PROKKA_run2 SRR9860122_unique.fasta
```

### Step 3: Download nf-core/phyloplace Pipeline
```bash
nextflow run nf-core/phyloplace -r 2.0.0 -profile test,docker --outdir results/phyloplace_output -c local.config
```

### Step 4: Run Pipeline
```bash
nextflow run nf-core/phyloplace -r 1.0.0 \
--id ajy_run1 \
--queryseqfile fasta/SRR9860122.fasta,fasta/ERR485020.fasta \
--refseqfile PROKKA_06042025_aligned.fna \
--refphylogeny ref_aligned.fasta \
--model LG+F+R6 \
--outdir output/phyloplace_results \
-profile docker \
-c memory_override.config
```

### Step 5: Quality Control with FastQC
```bash
fastqc fastq/SRR9860122.fastq fastq/ERR485020.fastq fastq/SRR10240887.fastq -o fastqc_results
```

### Step 6: Sequence Typing Using BLAST
```bash
blastn -query fasta/SRR9860122.fasta \
-db 16S_ribosomal_RNA \
-out seqtype_SRR9860122.txt \
-outfmt "6 qseqid sseqid pident length evalue bitscore stitle" \
-max_target_seqs 5 \
-num_threads 2
```

## Results and Visualizations

### Nextflow Workflow Report
- **Workflow ID**: `modest_yalow`
- **Execution Status**: Failed with exit code `7`
- **Failing Process**: `NFCORE_PHYLOPLACE:PHYLOPLACE:FASTA_NEWICK_EPANG_GAPPA:HMMER_UNALIGNREF`


### FastQC Summary: ERR485020.fastq
| Module | Status |
|--------|--------|
| Basic Statistics | PASS |
| Per base sequence quality | PASS |
| Per sequence quality scores | PASS |
| Per base sequence content | WARN |
| Sequence Duplication Levels | WARN |
| Adapter Content | PASS |

**Interpretation:** Warnings indicate minor issues (base content bias, some duplication). Data is usable.

## Discussion
- Annotation with **Prokka** completed successfully.
- Sequence QC with **FastQC** confirms suitable input quality.
- Sequence Typing via **BLAST+** gave useful matches.
- Pipeline execution failed. Troubleshooting required (e.g., memory/config/input format).

## Future Plan
- Re-run pipeline with revised config
- Check input formats and version compatibility
- Compare with tools like **IQ-TREE**, **BEAST**, **PhyloPhlAn**
- Perform extended analysis: AMR genes, virulence factors

## Limitations & Caveats
- Pipeline execution failed; no final phylogenetic tree was produced.
- Only 3 samples used, limiting analysis power.
- nf-core/phyloplace lacks extensive peer-reviewed use.

## References
- Ewels et al. (2020). *The nf-core framework*. Nature Biotechnology. https://doi.org/10.1038/s41587-020-0439-x
- Petit & Read (2020). *Bactopia pipeline*. mSystems. https://doi.org/10.1128/mSystems.00190-20
- nf-core/phyloplace GitHub: https://github.com/nf-core/phyloplace
